# Notas — Aula 2: Coleções e funções: o robô estruturado

Marco do robô: **v2 → v3 → v4**. A grade vira matriz 2D com obstáculos; posição e estado viram registros em `tuple` e `dict`; o script é refatorado em funções — todas com `robo` como primeiro argumento (semente do `self` da D2).

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**, ou `Kernel → Restart & Run All`) antes de pular para qualquer seção — a célula seguinte define constantes usadas no notebook inteiro. Se aparecer `NameError`, é sinal de que uma célula anterior não foi executada nesta sessão do kernel.


In [1]:
# Constantes usadas em todo o notebook — EXECUTE esta célula primeiro
LADO_GRADE = 10
DELTAS = {"LESTE": (1, 0), "NORTE": (0, 1), "OESTE": (-1, 0), "SUL": (0, -1)}


## 1. Lista: fatiamento (slicing) e alias vs. cópia

`lista[inicio:fim]` devolve uma **fatia** — uma lista nova com os elementos do índice `inicio` até `fim - 1` (o `fim` é exclusivo, igual em `range()`). Omitir `inicio` começa do 0; omitir `fim` vai até o final. `lista[:]` — os dois omitidos — é a fatia **completa**: uma cópia nova da lista inteira. A mesma sintaxe funciona em string (`"banana"[1:3]` → `"an"`).

Isso importa porque `log = trajetoria` **não copia** a lista — cria outro nome para o **mesmo objeto**: mudar uma muda a outra. Para uma cópia independente, use `trajetoria[:]` (a fatia completa que acabamos de ver).

No robô, isso importa sempre que você quiser guardar um "retrato" da trajetória num ponto do programa e continuar andando sem que esse retrato mude junto. Guarde essa regra — ela reaparece já na próxima seção, com uma casca a mais, na grade 2D.


In [2]:
frutas = ["maçã", "banana", "cereja", "uva"]
print(frutas[1:3])   # ['banana', 'cereja'] — índices 1 e 2, fim exclusivo
print(frutas[:2])    # ['maçã', 'banana']    — do início até o índice 1
print(frutas[2:])    # ['cereja', 'uva']     — do índice 2 até o fim
print(frutas[:])     # cópia da lista inteira

print("banana"[1:3])  # 'an' — mesma sintaxe em string


['banana', 'cereja']
['maçã', 'banana']
['cereja', 'uva']
['maçã', 'banana', 'cereja', 'uva']
an


In [3]:
trajetoria = [(0, 0), (3, 0), (5, 0)]

alias = trajetoria            # mesmo objeto
trajetoria.append((9, 0))
print(alias)                   # mudou junto!

copia = trajetoria[:]          # objeto novo, independente
trajetoria.append((9, 1))
print(copia)                   # não mudou


[(0, 0), (3, 0), (5, 0), (9, 0)]
[(0, 0), (3, 0), (5, 0), (9, 0)]


### Sua vez


In [4]:
# Cópia independente de `original`, feita com fatiamento
original = [(0, 0), (1, 1), (2, 2)]
backup = original[:]
original.append((99, 99))
print(original)
print(backup)


[(0, 0), (1, 1), (2, 2), (99, 99)]
[(0, 0), (1, 1), (2, 2)]


## 2. Lista 2D: a grade como lista de listas

Antes da grade, a notação que ela usa: `[expressão for variável in iterável]` cria uma lista nova — para cada valor que `variável` assume percorrendo `iterável`, calcula `expressão` e guarda o resultado. Quando o valor do `for` não importa — só queremos repetir N vezes — a convenção é chamar a variável de `_`.

A grade do robô é um tabuleiro — linhas e colunas. Em Python, representamos isso como uma **lista de listas**. Mas há uma armadilha clássica — e é o mesmo alias que acabamos de ver, disfarçado: `[[0] * LADO] * LADO` cria `LADO` referências para a **mesma** lista interna — mudar uma célula muda a linha inteira. A forma correta usa uma *list comprehension*, que cria um objeto novo a cada iteração.

No robô, a grade guarda obstáculos e células visitadas; a convenção é `grade[y][x]` (linha primeiro, coluna depois).


In [5]:
print([0 for _ in range(5)])      # [0, 0, 0, 0, 0]  — _ porque o valor do range não é usado
print([n**2 for n in range(5)])   # [0, 1, 4, 9, 16] — aqui n importa, é elevado ao quadrado


[0, 0, 0, 0, 0]
[0, 1, 4, 9, 16]


In [6]:
# Errado: todas as linhas são o MESMO objeto (aliasing)
LADO = 4
grade_errada = [[0] * LADO] * LADO
grade_errada[0][0] = 9
print(grade_errada)   # todas as linhas viram [9, 0, 0, 0]

# Certo: list comprehension cria LADO objetos independentes
grade_certa = [[0] * LADO for _ in range(LADO)]
grade_certa[0][0] = 9
print(grade_certa)    # só a primeira linha muda


[[9, 0, 0, 0], [9, 0, 0, 0], [9, 0, 0, 0], [9, 0, 0, 0]]
[[9, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]


In [7]:
# Aplicação no robô: grade 10x10, convenção grade[y][x]
grade = [[0] * LADO_GRADE for _ in range(LADO_GRADE)]
grade[2][3] = 1   # robô em (x=3, y=2)
print(f"grade[2][3] = {grade[2][3]}")


grade[2][3] = 1


### Sua vez


In [8]:
# Grade 5x5 (list comprehension) com a célula (x=4, y=1) marcada
LADO_TESTE = 5
grade_teste = [[0] * LADO_TESTE for _ in range(LADO_TESTE)]
grade_teste[1][4] = 7
print(grade_teste)


[[0, 0, 0, 0, 0], [0, 0, 0, 0, 7], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


## 3. Tupla: registro imutável

Tupla é como lista, mas **imutável** — não dá para reatribuir um elemento (`TypeError: 'tuple' object does not support item assignment`). Também é possível **desempacotar** os valores numa linha só: `x, y = posicao`.

No robô, a posição `(x, y)` e os deltas de movimento (`DELTAS['NORTE']`) são tuplas: um registro fixo que não muda depois de criado — se o robô se move, criamos uma posição **nova**, não editamos a antiga.


In [9]:
posicao = (3, 7)
x, y = posicao      # desempacotamento
print(f"x={x}, y={y}")

# posicao[0] = 5    # descomente para ver: TypeError (tupla não aceita atribuição)

dx, dy = DELTAS["NORTE"]
print(f"delta norte: dx={dx}, dy={dy}")


x=3, y=7
delta norte: dx=0, dy=1


### Sua vez


In [10]:
# Desempacotar leitura_sensor em direcao e livre
leitura_sensor = ("NORTE", True)
direcao, livre = leitura_sensor
print(f"{direcao} está livre: {livre}")


NORTE está livre: True


## 4. Dicionário: criação e acesso

Dicionário é uma **tabela de consulta**: uma chave dá acesso a um valor, em tempo O(1) — independente do tamanho da tabela (lista precisaria percorrer tudo, O(n)). `dict[chave]` lança `KeyError` se a chave não existir; `dict.get(chave, default)` devolve um valor seguro.

No robô, o dict vira tabela de obstáculos, tabela de rotação e o próprio estado do robô — tudo o que hoje é uma variável solta vai virar uma entrada de dict.


In [11]:
pontos = {"Alice": 10, "Bob": 7}
print(pontos["Alice"])          # 10
print(pontos.get("Zé", 0))      # 0 — default seguro, sem KeyError
print("Bob" in pontos)          # True
pontos["Alice"] = 12            # atualizar valor
for nome, pts in pontos.items():
    print(nome, pts)


10
0
True
Alice 12
Bob 7


### Sua vez


In [12]:
# Atualizar 'nivel' e ler 'temperatura' com default seguro
bateria = {"nivel": 80, "carregando": False}
bateria["nivel"] = 50
print(bateria)
print(bateria.get("temperatura", 20))


{'nivel': 50, 'carregando': False}
20


## 5. Dict como mapa esparso + contador de visitas

Para uma grade grande com poucos obstáculos, marcar cada obstáculo num dict (`{(x, y): True}`) é muito mais barato que alocar uma célula para cada posição da grade — só guardamos o que existe. O mesmo padrão serve para **contar**: `d[chave] = d.get(chave, 0) + 1` soma 1 a cada ocorrência, criando a chave na primeira vez que ela aparece.

No robô, isso vira o mapa de obstáculos e a contagem de células visitadas.


In [13]:
obstaculos = {(3, 2): True, (5, 5): True}
print((3, 2) in obstaculos)    # True
print((4, 2) in obstaculos)    # False

visitadas = {}
posicoes = [(0, 0), (1, 0), (0, 0), (2, 0), (0, 0)]
for pos in posicoes:
    visitadas[pos] = visitadas.get(pos, 0) + 1
print(visitadas)


True
False
{(0, 0): 3, (1, 0): 1, (2, 0): 1}


### Sua vez


In [14]:
# Contagem de comandos com o padrão d.get(k, 0) + 1
comandos_recebidos = ["AVANCAR", "GIRAR", "AVANCAR", "AVANCAR", "PARAR", "GIRAR"]
contagem = {}
for cmd in comandos_recebidos:
    contagem[cmd] = contagem.get(cmd, 0) + 1
print(contagem)


{'AVANCAR': 3, 'GIRAR': 2, 'PARAR': 1}


## 6. Dict como tabela de despacho + o robô como dict

Uma cascata de `if`/`elif` que só troca um valor por outro (ex.: girar o robô) pode virar uma **tabela de despacho**: um dict que mapeia entrada → saída diretamente. 8 ramos de `if/elif` viram 2 linhas de dict + 1 acesso.

Levando essa ideia adiante: o **estado inteiro do robô** também pode virar um dict só — `robo = {'x': 0, 'y': 0, 'direcao': 'LESTE', 'trajetoria': [(0, 0)]}`. Isso é um **proto-objeto**: na D2, esse dict vira uma `class Robo`, e cada chave vira um atributo (`self.x`, `self.direcao`, ...).


In [15]:
GIRAR_ESQ = {"LESTE": "NORTE", "NORTE": "OESTE", "OESTE": "SUL", "SUL": "LESTE"}

direcao = "NORTE"
nova_direcao = GIRAR_ESQ[direcao]
print(nova_direcao)   # OESTE — 1 linha, sem if/elif

robo = {"x": 0, "y": 0, "direcao": "LESTE", "trajetoria": [(0, 0)]}
print(f"Robô em ({robo['x']}, {robo['y']}), direção {robo['direcao']}")


OESTE
Robô em (0, 0), direção LESTE


### Sua vez


In [16]:
# Girar à direita usando a tabela de despacho GIRAR_DIR
GIRAR_DIR = {"LESTE": "SUL", "SUL": "OESTE", "OESTE": "NORTE", "NORTE": "LESTE"}
robo = {"x": 0, "y": 0, "direcao": "NORTE", "trajetoria": [(0, 0)]}
robo["direcao"] = GIRAR_DIR[robo["direcao"]]
print(robo["direcao"])


LESTE


## 7. Funções: `def`, parâmetros e retorno

`def nome(parâmetros):` define uma função; `return` entrega um valor ao chamador e encerra a execução. Sem `return`, a função devolve `None`.

No robô, a checagem "essa posição é válida?" — que se repetia várias vezes no v2 — vira uma função só, chamada de todo lugar que precisar dela.


In [17]:
def posicao_valida(x, y):
    return 0 <= x < LADO_GRADE and 0 <= y < LADO_GRADE

print(posicao_valida(5, 3))     # True
print(posicao_valida(10, 0))    # False — 10 == LADO_GRADE, limite exclusivo


True
False


### Sua vez


In [18]:
# esta_na_borda: True se x ou y está numa borda da grade
def esta_na_borda(x, y):
    return (x == 0 or x == LADO_GRADE - 1 or
            y == 0 or y == LADO_GRADE - 1)

print(esta_na_borda(0, 5))   # True
print(esta_na_borda(5, 5))   # False


True
False


## 8. `return` vs. `print` — o maior ponto de confusão

`print` mostra algo na tela; `return` entrega um valor que o **chamador** pode usar. Uma função que só imprime devolve `None` para quem a chamou — mesmo que pareça ter "funcionado".

No robô, quase toda função (`sensor_frente`, `avancar`) precisa **retornar** um resultado, porque quem chama decide o que fazer com ele (mover ou não, imprimir uma mensagem ou não).


In [19]:
def saudacao_com_bug(nome):
    print(f"Olá, {nome}!")   # imprime, mas não retorna

resultado = saudacao_com_bug("Alice")
print(f"Resultado: {resultado}")   # None — a função não usou return

def saudacao(nome):
    return f"Olá, {nome}!"

resultado = saudacao("Alice")
print(f"Resultado: {resultado}")   # Olá, Alice!


Olá, Alice!
Resultado: None
Resultado: Olá, Alice!


### Sua vez


In [20]:
# formatar_posicao RETORNA a string em vez de imprimir direto
def formatar_posicao(x, y):
    return f"Robô em ({x}, {y})"

msg = formatar_posicao(3, 4)
print(msg)


Robô em (3, 4)


## 9. Parâmetros com valor padrão + argumento padrão mutável (armadilha)

Um parâmetro pode ter um **valor padrão**, usado quando quem chama não passa esse argumento: `def nome(parametro=valor_padrao):`. Parâmetro com default vira opcional.

**Nunca** use lista ou dict como esse valor padrão. A lista/dict padrão é criada **uma vez só**, na definição da função, e reaproveitada em todas as chamadas que não passam esse argumento — o que causa um bug sutil: dados de uma chamada "vazam" para a próxima.

O conserto: use `None` como padrão e crie a lista/dict **dentro** da função.


In [21]:
def saudacao_com_padrao(nome, cumprimento="Olá"):
    return f"{cumprimento}, {nome}!"

print(saudacao_com_padrao("Ana"))         # "Olá, Ana!" — usa o padrão
print(saudacao_com_padrao("Ana", "Oi"))   # "Oi, Ana!"  — sobrescreve o padrão


Olá, Ana!
Oi, Ana!


In [22]:
def registrar(evento, lista_eventos=[]):    # BUG: mutável como default
    lista_eventos.append(evento)
    return lista_eventos

print(registrar("A"))   # ['A'] — parece OK
print(registrar("B"))   # ['A', 'B'] — mesma lista da chamada anterior!

def registrar_corrigido(evento, lista_eventos=None):
    if lista_eventos is None:
        lista_eventos = []
    lista_eventos.append(evento)
    return lista_eventos

print(registrar_corrigido("A"))   # ['A']
print(registrar_corrigido("B"))   # ['B'] — lista nova a cada chamada


['A']
['A', 'B']
['A']
['B']


### Sua vez


In [23]:
# criar_leitura corrigida: None + criação dentro da função
def criar_leitura(direcao, historico=None):
    if historico is None:
        historico = []
    historico.append(direcao)
    return historico

print(criar_leitura("NORTE"))
print(criar_leitura("LESTE"))


['NORTE']
['LESTE']


## 10. O padrão do robô: `robo` como primeiro argumento

Quando o estado do robô virou um dict (seção 6), toda função que opera sobre ele passou a receber esse dict como argumento — e, como o dict é **mutável**, as alterações feitas dentro da função refletem fora dela (não precisa retornar um "novo robô").

Reparem no padrão: `sensor_frente(robo, obstaculos)`, `avancar(robo, obstaculos)`, `girar(robo, lado)` — todas começam com `robo`. Essa é a **semente do `self`**: na D2, `avancar(robo, obs)` vira `robo.avancar(obs)`, onde `self` é exatamente o `robo` de hoje.


In [24]:
def sensor_frente(robo, obstaculos):
    dx, dy = DELTAS[robo["direcao"]]
    nx, ny = robo["x"] + dx, robo["y"] + dy
    return posicao_valida(nx, ny) and (nx, ny) not in obstaculos

def avancar(robo, obstaculos):
    if sensor_frente(robo, obstaculos):
        dx, dy = DELTAS[robo["direcao"]]
        robo["x"] += dx
        robo["y"] += dy
        robo["trajetoria"].append((robo["x"], robo["y"]))
        return True
    return False

robo = {"x": 0, "y": 0, "direcao": "LESTE", "trajetoria": [(0, 0)]}
obstaculos = {(3, 2): True}
print(avancar(robo, obstaculos))   # True
print(robo["x"], robo["y"])        # 1 0


True
1 0


### Sua vez


In [25]:
# girar(robo, lado): atualiza robo['direcao'] in-place
GIRAR_ESQ = {"LESTE": "NORTE", "NORTE": "OESTE", "OESTE": "SUL", "SUL": "LESTE"}
GIRAR_DIR = {"LESTE": "SUL", "SUL": "OESTE", "OESTE": "NORTE", "NORTE": "LESTE"}

def girar(robo, lado):
    if lado == "ESQ":
        robo["direcao"] = GIRAR_ESQ[robo["direcao"]]
    elif lado == "DIR":
        robo["direcao"] = GIRAR_DIR[robo["direcao"]]

robo2 = {"x": 0, "y": 0, "direcao": "LESTE", "trajetoria": [(0, 0)]}
girar(robo2, "ESQ")
print(robo2["direcao"])   # NORTE


NORTE


## Para aprofundar

- **Dicionários:** https://www.w3schools.com/python/python_dictionaries.asp
- **Listas aninhadas (matrizes 2D):** https://realpython.com/python-lists-tuples/
- **Aliasing e cópia:** https://realpython.com/copying-python-objects/
- **Tuplas:** https://www.w3schools.com/python/python_tuples.asp
- **Funções — definição, parâmetros, retorno:** https://www.w3schools.com/python/python_functions.asp
- **Argumento padrão mutável (anti-padrão):** https://docs.python.org/3/faq/programming.html#why-are-default-values-shared-between-objects
